# Notebook 02: Shoulder Region Segmentation with U-Net++

Before classification, the pipeline isolates the shoulder joint from the surrounding background.
This notebook explains why this cropping step matters, how the U-Net++ segmentation model works,
and how the predicted mask is converted into a tight, square crop centered on the humeral head.

**What you will learn:**
- Why cropping before classification improves accuracy
- The U-Net++ architecture with an EfficientNet-B2 encoder
- How to load a PyTorch checkpoint and handle state_dict key mismatches
- How to run inference and convert a probability map into a binary mask
- The crop geometry: bounding box extraction, head-bias offset, and square padding

**Source code references:**
- `Backend/ai_logic.py` — `crop_with_unet()`, `_crop_with_unet_impl()`
- `Backend/model_loader.py` — U-Net checkpoint loading

> **Note on model weights:** The U-Net checkpoint is downloaded at runtime from Google Drive
> via `Backend/config.py`. If you are running this notebook without the weights, inference cells
> will be skipped and a fallback (center crop) will be shown instead.

## 1. Why Crop Before Classification?

A raw shoulder X-ray contains a large amount of contextually irrelevant information:
- The spine, ribs, clavicle, and scapula body
- Radiopaque markers and patient labels
- Dark background regions

If a CNN is trained on full-resolution images, it must learn to identify the relevant region
among all this noise. The model wastes capacity on uninformative areas and may latch onto
confounding features (for example, markers placed near calcifications).

Cropping to the joint region has two concrete benefits:

1. **Higher effective resolution on the region of interest.** After resizing the crop to the
   model's input size (300x300 or 512x512), each pixel covers less anatomical area, giving
   the network more detail to work with.

2. **Reduced domain shift.** The crop removes scanner-specific annotations and borders,
   making the representation more consistent across different acquisition systems.

The segmentation model is a **U-Net++** (nested U-Net) with an **EfficientNet-B2** encoder,
trained to predict a binary mask covering the humeral head and proximal humerus.

## 2. Architecture: U-Net++ with EfficientNet-B2

### U-Net (baseline)
U-Net (Ronneberger et al., 2015) is a fully convolutional encoder-decoder architecture designed
for biomedical image segmentation. It consists of:
- An **encoder** (contracting path) that progressively downsamples the input using convolutions
  and pooling, extracting hierarchical features.
- A **decoder** (expanding path) that upsamples the feature maps back to the original resolution.
- **Skip connections** that concatenate encoder features directly to the corresponding decoder
  level, preserving spatial detail lost during downsampling.

### U-Net++
U-Net++ (Zhou et al., 2018) extends this design by introducing **dense nested skip connections**.
Instead of a single skip connection per level, intermediate nodes are inserted that re-encode the
feature maps before passing them to the decoder. This redesign:
- Reduces the semantic gap between encoder and decoder feature maps
- Allows the model to aggregate features at multiple scales simultaneously
- Consistently outperforms standard U-Net on medical segmentation benchmarks

### EfficientNet-B2 encoder
Rather than training a random encoder from scratch, the model uses **EfficientNet-B2** as a
pretrained backbone. EfficientNet scales network width, depth, and resolution in a principled
way (compound coefficient). The B2 variant offers a good trade-off between accuracy and
computational cost for this task.

Using a pretrained encoder reduces the amount of labeled data required and typically leads
to faster convergence and better generalization.

The model is implemented using the `segmentation_models_pytorch` (smp) library.

## 3. Imports and Setup

In [ ]:
import numpy as np
import cv2
import torch
import segmentation_models_pytorch as smp
import matplotlib.pyplot as plt
import pydicom
from pydicom.pixel_data_handlers.util import apply_voi_lut
import os

DICOM_PATH  = "../Frontend/Images/I_AP (1).dcm"
UNET_WEIGHTS = None  # Set to a local path if you have the checkpoint, e.g. "../Backend/unet.ckpt"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# Load and preprocess the DICOM (from Notebook 01)
def load_dicom_uint8(path):
    ds = pydicom.dcmread(path)
    if "WindowWidth" in ds and "WindowCenter" in ds:
        img = apply_voi_lut(ds.pixel_array, ds)
    else:
        img = ds.pixel_array.astype(float)
    if getattr(ds, "PhotometricInterpretation", "") == "MONOCHROME1":
        img = np.amax(img) - img
    img = img - np.min(img)
    if np.max(img) != 0:
        img = img / np.max(img)
    return (img * 255).astype(np.uint8)

img_uint8 = load_dicom_uint8(DICOM_PATH)
print(f"Image loaded: shape={img_uint8.shape}, dtype={img_uint8.dtype}")

plt.figure(figsize=(5, 5))
plt.imshow(img_uint8, cmap="gray")
plt.title("Input image (full resolution)")
plt.axis("off")
plt.tight_layout()
plt.show()

## 4. Loading the PyTorch Checkpoint

The segmentation model is serialized as a `.ckpt` file (PyTorch Lightning checkpoint format).
The state dict may contain key prefixes added by the training framework (`model.` or `net.`)
that do not match the raw `smp.UnetPlusPlus` key names. These must be stripped before loading.

In [ ]:
unet_model = None

if UNET_WEIGHTS and os.path.exists(UNET_WEIGHTS):
    # Build model architecture (encoder weights are None because we load from checkpoint)
    unet_model = smp.UnetPlusPlus(
        encoder_name="efficientnet-b2",
        encoder_weights=None,
        in_channels=3,
        classes=1,
    )

    checkpoint = torch.load(UNET_WEIGHTS, map_location="cpu")

    # Lightning checkpoints wrap the weights under a 'state_dict' key
    state_dict = checkpoint.get("state_dict", checkpoint)

    # Strip framework-added prefixes so keys match the smp model
    cleaned = {
        k.replace("model.", "").replace("net.", ""): v
        for k, v in state_dict.items()
    }

    unet_model.load_state_dict(cleaned, strict=False)
    unet_model.to(device).eval()
    print("U-Net++ loaded successfully.")
else:
    print("No checkpoint path provided. A center-crop fallback will be used for visualization.")

## 5. Running Segmentation Inference

The inference pipeline converts the grayscale image to a 3-channel float tensor,
resizes it to 512x512 (the model's training resolution), and applies a sigmoid
activation to the output logit to obtain a probability map in [0, 1].

In [ ]:
def run_unet_inference(img_uint8, model, device, mask_thr=0.5):
    """
    Returns:
        prob_map  - float array [0,1], shape (512,512)
        mask_bin  - uint8 binary mask, shape (512,512), values 0 or 255
    """
    img_512 = cv2.resize(img_uint8, (512, 512), interpolation=cv2.INTER_AREA)
    img_rgb = cv2.cvtColor(img_512, cv2.COLOR_GRAY2RGB).astype("float32") / 255.0

    # HWC -> CHW, add batch dimension
    tensor = torch.from_numpy(img_rgb.transpose(2, 0, 1)).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(tensor)                        # shape: (1, 1, 512, 512)
        prob   = torch.sigmoid(logits)[0, 0]         # shape: (512, 512)
        prob_map = prob.detach().cpu().numpy()

    mask_bin = (prob_map > mask_thr).astype(np.uint8) * 255
    return prob_map, mask_bin


if unet_model is not None:
    prob_map, mask_bin = run_unet_inference(img_uint8, unet_model, device)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(img_uint8, cmap="gray")
    axes[0].set_title("Original (full res)")
    axes[0].axis("off")

    axes[1].imshow(prob_map, cmap="hot", vmin=0, vmax=1)
    axes[1].set_title("U-Net probability map")
    axes[1].axis("off")

    axes[2].imshow(mask_bin, cmap="gray")
    axes[2].set_title(f"Binary mask (threshold=0.5)")
    axes[2].axis("off")

    plt.tight_layout()
    plt.show()
else:
    print("Skipping inference visualization (no model weights).")

## 6. Converting the Mask to a Crop

The binary mask identifies the segmented region, but the final goal is a square crop of the
original image centered on the humeral head. The crop geometry is computed as follows:

1. **Find the largest contour** in the 512x512 binary mask. This corresponds to the main
   segmented region (the humeral head and proximal humerus).

2. **Scale the bounding box back** to the original image resolution using the scale factors
   `scale_x = orig_w / 512` and `scale_y = orig_h / 512`.

3. **Apply a head-bias offset.** The humeral head is located near the top of the bounding box,
   not at its center. Setting the crop center `y` to `bbox_top + 20% of bbox_height` shifts
   the crop upward toward the relevant anatomy.

4. **Determine the side length.** The crop side is `max(width, height) * 1.1` to include a
   small margin around the detected region.

5. **Clamp to image boundaries** and apply symmetric zero-padding to guarantee a perfect
   square even near the image edges.

In [ ]:
def mask_to_crop(img_uint8, mask_bin, head_bias_ratio=0.2, margin=1.1):
    orig_h, orig_w = img_uint8.shape
    scale_x = orig_w / 512.0
    scale_y = orig_h / 512.0

    contours, _ = cv2.findContours(mask_bin, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if not contours:
        # Fallback: center crop
        cx, cy = orig_w // 2, orig_h // 2
        side_size = min(orig_w, orig_h) // 2
    else:
        c = max(contours, key=cv2.contourArea)
        x, y, w, h = cv2.boundingRect(c)
        x = int(x * scale_x)
        y = int(y * scale_y)
        w = int(w * scale_x)
        h = int(h * scale_y)

        cx = x + w // 2
        cy = int(y + h * head_bias_ratio)   # shift toward the top of the bounding box
        side_size = int(max(w, h) * margin)

    half = side_size // 2
    x0, y0 = cx - half, cy - half
    x1, y1 = x0 + side_size, y0 + side_size

    # Clamp to image boundaries
    if x0 < 0:      x1 += abs(x0); x0 = 0
    if x1 > orig_w: x0 -= (x1 - orig_w); x1 = orig_w
    if y0 < 0:      y1 += abs(y0); y0 = 0
    if y1 > orig_h: y0 -= (y1 - orig_h); y1 = orig_h
    x0, y0 = max(0, x0), max(0, y0)
    x1, y1 = min(orig_w, x1), min(orig_h, y1)

    crop = img_uint8[y0:y1, x0:x1]

    # Pad to square if the crop is not perfectly square after clamping
    ch, cw = crop.shape
    if ch != cw:
        diff = abs(ch - cw)
        p1, p2 = diff // 2, diff - diff // 2
        if ch < cw:
            crop = np.pad(crop, ((p1, p2), (0, 0)), mode="constant")
        else:
            crop = np.pad(crop, ((0, 0), (p1, p2)), mode="constant")

    return crop, (x0, y0, x1, y1)


if unet_model is not None:
    crop, (x0, y0, x1, y1) = mask_to_crop(img_uint8, mask_bin)

    fig, axes = plt.subplots(1, 2, figsize=(11, 5))

    # Draw bounding box on original
    display_orig = cv2.cvtColor(img_uint8, cv2.COLOR_GRAY2RGB)
    cv2.rectangle(display_orig, (x0, y0), (x1, y1), (0, 220, 0), 4)
    axes[0].imshow(display_orig)
    axes[0].set_title("Crop region on original image")
    axes[0].axis("off")

    axes[1].imshow(crop, cmap="gray")
    axes[1].set_title(f"Crop result ({crop.shape[1]}x{crop.shape[0]} px)")
    axes[1].axis("off")

    plt.tight_layout()
    plt.show()
else:
    # Show a simple center crop as fallback
    h, w = img_uint8.shape
    side = min(h, w) // 2
    cx, cy = w // 2, h // 2
    crop_fallback = img_uint8[cy - side//2 : cy + side//2, cx - side//2 : cx + side//2]

    fig, axes = plt.subplots(1, 2, figsize=(11, 5))
    axes[0].imshow(img_uint8, cmap="gray")
    axes[0].set_title("Original image")
    axes[0].axis("off")
    axes[1].imshow(crop_fallback, cmap="gray")
    axes[1].set_title("Center crop fallback (no model weights)")
    axes[1].axis("off")
    plt.tight_layout()
    plt.show()

## 7. Inference Caching in Production

In the deployed Flask application, a user request may trigger multiple predictions
(for example, SEG_300 and Grad-CAM for the same image in quick succession).
Running the U-Net twice for the same image wastes time and GPU memory.

The production code in `Backend/ai_logic.py` implements a **TTL cache** (Time-To-Live,
180 seconds) keyed by a fast image fingerprint:

```python
def _crop_fingerprint(img):
    sample = img.ravel()[:4096].tobytes()
    return hashlib.md5(sample + str(img.shape).encode()).hexdigest()
```

Only the first 4096 bytes of the flattened array plus the shape are hashed. This is fast
and sufficient to distinguish between different images with very high probability. The full
MD5 of a large medical image would add unnecessary latency.

## 8. Summary

The shoulder segmentation step produces a square crop of the humeral head region from a
full-resolution X-ray. The key design decisions are:

| Decision | Reason |
|----------|--------|
| U-Net++ over standard U-Net | Dense nested skip connections reduce the semantic gap between encoder and decoder features |
| EfficientNet-B2 encoder | Pretrained features reduce labeled data requirements and speed up convergence |
| 512x512 inference resolution | Balances detail and computational cost |
| Head-bias crop center (20%) | The humeral head sits near the top of the segmented region, not at its centroid |
| 10% margin around bounding box | Includes surrounding soft tissue context needed by the classifier |
| Square padding | All downstream models expect square inputs |

The crop produced here is the input to **Notebook 03** (tendinopathy classification) and
**Notebook 04** (YOLO anatomical detection).